In [35]:
import os
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
import google.genai as genai

In [36]:
load_dotenv()
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("Please set GOOGLE_API_KEY in your environment or .env file.")
client = genai.Client(api_key=api_key)

In [37]:
class cricketState(TypedDict):
    runs:int
    balls:int
    fours:int
    six:int

    sr:float
    bpb:float
    bp:float
    summary:str




In [38]:
def srcal(state:cricketState)->cricketState:
    sr1 = (state['runs']/state['balls'])/100
    return {'sr':sr1}
def bpbcal(state:cricketState)->cricketState:
    bpb1 = state['balls']/(state['fours'] + state['six'])
    return {'bpb':bpb1}
def bpcal(state:cricketState)->cricketState:
    bp1 = (((state['fours']*4) + (state['six']*6))/state['runs'])*100
    return {'bp':bp1}
def summarycal(state:cricketState)->cricketState:
    sum = f"""strike rate-{state['sr']} \n balls per boundary - {state['bpb']} \n boundary percent - {state['bp']}"""
    return {'summary':sum}



In [39]:
graph = StateGraph(cricketState)
# node creation
graph.add_node('sr',srcal)
graph.add_node('bp',bpcal)
graph.add_node('bpb',bpbcal)
graph.add_node('summary',summarycal)
# edge creation at level 1
graph.add_edge(START,'sr')
graph.add_edge(START,'bpb')
graph.add_edge(START,'bp')
# edge creation at level 2
graph.add_edge('sr','summary')
graph.add_edge('bp','summary')
graph.add_edge('bpb','summary')
# end 
graph.add_edge('summary',END)

# complile
workflow = graph.compile()

In [40]:
initial_state = {'runs':100,
                 'balls':50,
                 'fours':6,
                 'six':4}
final_state = workflow.invoke(initial_state)
print(final_state)

{'runs': 100, 'balls': 50, 'fours': 6, 'six': 4, 'sr': 0.02, 'bpb': 5.0, 'bp': 48.0, 'summary': 'strike rate-0.02 \n balls per boundary - 5.0 \n boundary percent - 48.0'}
